In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.customer360;

In [0]:
%sql
use catalog workspace;
use schema customer360;

select current_database(), current_catalog(), current_schema();

In [0]:
%python
customers = [
    (1001, "John Smith", "Dallas", "TX", "Wireless", "Active"),
    (1002, "Mary Jones", "Plano", "TX", "Fiber", "Active"),
    (1003, "Steve Brown", "Frisco", "TX", "Wireless", "Suspended"),
    (1004, "Sarah Wilson", "Irving", "TX", "Fiber", "Inactive"),
    (1005, "David Lee", "McKinney", "TX", "Wireless", "Active")
]

columns = ["custId", "custName", "city", "state", "product", "status"]

dfCustomers = spark.createDataFrame(customers,columns)

dfCustomers.write.mode("overwrite").saveAsTable("customers")
display(dfCustomers)



In [0]:
%sql
-- show customer table
select * from customer360.customers;

-- show customers with active status
select * from customer360.customers where status = 'Active';

-- order the custName 
select * from customer360.customers order by custName;

select * from customer360.customers order by custName desc;

In [0]:
%python
# select only custName and Product columns from customers table
display(dfCustomers.select("custName","product"))

# display only ACTIVE records
display(dfCustomers.filter("status = 'Active'"))

#order by
display(dfCustomers.orderBy(dfCustomers.custId.desc()))
       

In [0]:
%python
# Import lit function to create literal/constant values
from pyspark.sql.functions import lit

# Add a new column 'country' with constant value 'USA' to dfCustomers
# withColumn() creates a NEW DataFrame - must reassign to keep the change
dfCustomers = dfCustomers.withColumn("country", lit("USA"))

# Save the updated DataFrame to the customers table
# mode("overwrite") - replaces existing table data
# option("mergeSchema", "true") - allows adding new columns to existing table schema
dfCustomers.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("customers")

# Display the updated DataFrame with the new country column
display(dfCustomers)

In [0]:
%python
dfCustomers = dfCustomers.drop("state")
dfCustomers.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("customers")

display(dfCustomers)

In [0]:
%sql
select * from customer360.customers;

In [0]:
%python
display(dfCustomers.select("product").distinct())


In [0]:
%python
from pyspark.sql.functions import when

dfCustomers = dfCustomers.withColumn("custTier", when(dfCustomers["product"] == "Fiber", "Premium").otherwise("Standard"))
dfCustomers.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("customers")

display(dfCustomers)